In [33]:
# Install dependency (run once)
!pip install cryptography

import sqlite3
from cryptography.fernet import Fernet
import os

# --- KEY MANAGEMENT (SAFE) ---
if not os.path.exists("secret.key"):
    with open("secret.key", "wb") as f:
        f.write(Fernet.generate_key())

def load_key():
    return open("secret.key", "rb").read()

cipher = Fernet(load_key())

# --- DATABASE SETUP ---
conn = sqlite3.connect("secure_data.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    username TEXT PRIMARY KEY,
    password BLOB
)
""")
conn.commit()

# --- FUNCTIONS ---
def encrypt_password(password):
    return cipher.encrypt(password.encode())

def decrypt_password(encrypted_password):
    return cipher.decrypt(encrypted_password).decode()

def register_user(username, password):
    cursor.execute("SELECT * FROM users WHERE username=?", (username,))
    if cursor.fetchone():
        print("⚠️ User already exists!")
        return

    encrypted_pw = encrypt_password(password)
    cursor.execute("INSERT INTO users VALUES (?, ?)", (username, encrypted_pw))
    conn.commit()
    print("✅ User registered securely!")

def login_user(username, password):
    cursor.execute("SELECT password FROM users WHERE username=?", (username,))
    result = cursor.fetchone()

    if not result:
        print("❌ User not found!")
        return

    try:
        if decrypt_password(result[0]) == password:
            print("✅ Login successful!")
        else:
            print("❌ Incorrect password!")
    except:
        print("❌ Security error!")

In [34]:
# Register user
register_user("admin", "secure123")

# Try duplicate
register_user("admin", "secure123")

# Correct login
login_user("admin", "secure123")

# Wrong password
login_user("admin", "wrongpass")

# SQL Injection attempt (fails safely)
login_user("' OR 1=1 --", "hack")

⚠️ User already exists!
⚠️ User already exists!
❌ Security error!
❌ Security error!
❌ User not found!
